# 02 — Limpieza / Filtrado: Chihuahua + causa de suicidio (Etapa 2)

**Proyecto:** chihuahua-suicide-seasonality-replication
**Etapa:** Limpieza/Filtrado

Este notebook parte de los archivos crudos nacionales (sin filtrar) generados por
`01_download_extract_mortalidad.ipynb` en `data/interim/defunciones_{año}_crudo.parquet`
(17 años, 2008-2024, ya armonizados en tipo de dato pero SIN filtrar por geografía ni causa).

Aplica dos filtros documentados en `docs/DATA_DICTIONARY.md`:
1. **Geográfico**: `ENT_RESID == "08"` (Chihuahua, **residencia habitual** — decisión ya tomada).
2. **Causa de defunción — suicidio**, con **dos filtros independientes** para validación cruzada:
   - `CAUSA_DEF` en el rango CIE-10 X60-X84 (lesiones autoinfligidas intencionalmente).
   - Campo armonizado `PRESUNTO` (2005-2021) / `TIPO_DEFUN` (2022-2024) `== 3` ("Suicidio").

⚠️ Este notebook está pensado para correr en tu computadora, donde ya tienes
`data/interim/` poblado por el notebook 01. No se ejecuta en este entorno de Claude.

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")  # este notebook vive en notebooks/, el proyecto está un nivel arriba
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2008, 2025))  # 2008-2024 (réplica 2008-2018 + extensión 2019-2024)

## 1. Carga de los 17 años crudos (nacional, sin filtrar) generados en el notebook 01

In [ ]:
dfs = []
missing_years = []

for year in YEARS:
    path = INTERIM_DIR / f"defunciones_{year}_crudo.parquet"
    if not path.exists():
        missing_years.append(year)
        continue
    df_year = pd.read_parquet(path)
    dfs.append(df_year)

if missing_years:
    print(f"⚠️ Faltan archivos crudos para los años: {missing_years}. "
          f"Corre primero el notebook 01 para esos años.")

df_nacional = pd.concat(dfs, ignore_index=True)
print(f"Total nacional cargado (2008-2024, sin filtrar): {len(df_nacional):,} filas")
print(f"Columnas: {len(df_nacional.columns)}")

# Filas por año, como chequeo rápido de que la carga esté completa
print()
print("Filas por año (nacional, sin filtrar):")
print(df_nacional["ANIO_OCUR"].value_counts().sort_index())

## 2. Filtro geográfico — Chihuahua, residencia habitual (`ENT_RESID == "08"`)

`ENT_RESID` quedó guardado como texto con ceros a la izquierda (ancho 2) desde la
armonización de tipos del notebook 01, así que comparamos contra el string `"08"`,
no contra el entero `8`.

In [ ]:
print("Valores únicos de ENT_RESID (primeros 20, para confirmar formato):")
print(sorted(df_nacional["ENT_RESID"].dropna().unique())[:20])

df_chihuahua = df_nacional[df_nacional["ENT_RESID"] == "08"].copy()

print()
print(f"Filas tras filtro geográfico (Chihuahua, residencia habitual): {len(df_chihuahua):,}")
print(f"  ({len(df_chihuahua) / len(df_nacional) * 100:.3f}% del total nacional)")

print()
print("Filas por año (Chihuahua, todas las causas de defunción):")
print(df_chihuahua["ANIO_OCUR"].value_counts().sort_index())

## 3. Filtro de causa — suicidio, con dos filtros independientes

### 3.1 Campo armonizado de tipo de defunción (`PRESUNTO` 2005-2021 / `TIPO_DEFUN` 2022-2024)

Ambos campos coexisten como columnas tras el concat (una queda NaN según el periodo).
El código **3 = Suicidio en ambos sistemas** (confirmado en `docs/DATA_DICTIONARY.md`),
así que el filtro `== 3` es válido sin mapeo condicional adicional, PERO construimos
la columna armonizada explícitamente para dejar documentada la equivalencia y evitar
cualquier ambigüedad en pasos posteriores.

In [ ]:
if "PRESUNTO" not in df_chihuahua.columns:
    df_chihuahua["PRESUNTO"] = pd.NA
if "TIPO_DEFUN" not in df_chihuahua.columns:
    df_chihuahua["TIPO_DEFUN"] = pd.NA

df_chihuahua["tipo_defuncion_cod"] = df_chihuahua["PRESUNTO"].where(
    df_chihuahua["PRESUNTO"].notna(), df_chihuahua["TIPO_DEFUN"]
)

# Chequeo: cada fila debe tener el dato en PRESUNTO (años <=2021) o en TIPO_DEFUN (>=2022),
# nunca en ambos ni en ninguno.
sin_dato = df_chihuahua["tipo_defuncion_cod"].isna().sum()
en_ambos = (df_chihuahua["PRESUNTO"].notna() & df_chihuahua["TIPO_DEFUN"].notna()).sum()
print(f"Filas sin PRESUNTO ni TIPO_DEFUN: {sin_dato}")
print(f"Filas con PRESUNTO y TIPO_DEFUN a la vez (no debería pasar): {en_ambos}")

filtro_tipo_defuncion = df_chihuahua["tipo_defuncion_cod"] == 3

### 3.2 Filtro por CIE-10 (`CAUSA_DEF` en X60-X84)

`CAUSA_DEF` quedó como texto de 4 caracteres (ancho documentado en el notebook 01,
ej. `"X600"` = X60.0). Comparamos por los primeros 3 caracteres (letra + 2 dígitos)
contra el rango X60-X84.

In [ ]:
print("Ejemplos de valores de CAUSA_DEF (primeros 20 únicos, para confirmar formato):")
print(sorted(df_chihuahua["CAUSA_DEF"].dropna().unique())[:20])

causa_prefijo = df_chihuahua["CAUSA_DEF"].str[:3]
filtro_cie10 = causa_prefijo.between("X60", "X84")

### 3.3 Comparación de ambos filtros (validación cruzada)

Si ambos filtros no coinciden exactamente, hay que revisar los casos discordantes
antes de decidir cuál usar (o si se combina con AND/OR) — no asumir cuál es "el bueno".

In [ ]:
n_cie10 = filtro_cie10.sum()
n_tipo_defuncion = filtro_tipo_defuncion.sum()
n_interseccion = (filtro_cie10 & filtro_tipo_defuncion).sum()
n_solo_cie10 = (filtro_cie10 & ~filtro_tipo_defuncion).sum()
n_solo_tipo_defuncion = (~filtro_cie10 & filtro_tipo_defuncion).sum()

print(f"Filtro CIE-10 (CAUSA_DEF X60-X84):           {n_cie10:,} filas")
print(f"Filtro tipo de defunción (== 3, Suicidio):    {n_tipo_defuncion:,} filas")
print(f"Intersección (ambos filtros coinciden):       {n_interseccion:,} filas")
print(f"Solo CIE-10, NO tipo de defunción:            {n_solo_cie10:,} filas")
print(f"Solo tipo de defunción, NO CIE-10:            {n_solo_tipo_defuncion:,} filas")

if n_solo_cie10 > 0 or n_solo_tipo_defuncion > 0:
    print()
    print("⚠️ Hay discrepancias entre los dos filtros. Revisa una muestra antes de decidir:")
    cols_revision = ["ANIO_OCUR", "ENT_RESID", "CAUSA_DEF", "PRESUNTO", "TIPO_DEFUN",
                      "tipo_defuncion_cod", "anio_archivo_origen"]
    cols_revision = [c for c in cols_revision if c in df_chihuahua.columns]

    if n_solo_cie10 > 0:
        print()
        print(f"Muestra: CIE-10 dice suicidio pero tipo de defunción no (hasta 10 filas):")
        print(df_chihuahua.loc[filtro_cie10 & ~filtro_tipo_defuncion, cols_revision].head(10))

    if n_solo_tipo_defuncion > 0:
        print()
        print(f"Muestra: tipo de defunción dice suicidio pero CIE-10 no (hasta 10 filas):")
        print(df_chihuahua.loc[~filtro_cie10 & filtro_tipo_defuncion, cols_revision].head(10))

### 3.4 Decisión de filtro final

Por defecto usamos la **intersección** (ambos filtros de acuerdo) como el criterio más
conservador y confiable para el dataset curado principal. Si el paso 3.3 muestra
discrepancias, compártelas antes de continuar — puede ser necesario ajustar esta
decisión (ej. usar unión, o investigar un tercer criterio) y lo documentamos en el
reporte de la Etapa 2.

In [ ]:
df_suicidios_chihuahua = df_chihuahua[filtro_cie10 & filtro_tipo_defuncion].copy()
print(f"Dataset final (Chihuahua + suicidio, ambos filtros de acuerdo): "
      f"{len(df_suicidios_chihuahua):,} filas, 2008-2024")

## 4. Validación contra el total del artículo original (3,572 casos, 2008-2018)

In [ ]:
df_replica = df_suicidios_chihuahua[
    df_suicidios_chihuahua["ANIO_OCUR"].astype(int).between(2008, 2018)
]
total_replica = len(df_replica)
total_articulo = 3572
diferencia = total_replica - total_articulo

print(f"Casos 2008-2018 (nuestro filtrado): {total_replica:,}")
print(f"Casos 2008-2018 (artículo original): {total_articulo:,}")
print(f"Diferencia: {diferencia:+,} ({diferencia / total_articulo * 100:+.2f}%)")

print()
print("Desglose por año (2008-2018) — compáralo contra la Tabla 1 del artículo si la tienes a mano:")
print(df_replica["ANIO_OCUR"].value_counts().sort_index())

print()
print("Desglose por sexo (para chequeo rápido contra el 80%/20% reportado en el artículo):")
print(df_replica["SEXO"].value_counts(normalize=True) * 100)

## 5. Guardar el dataset filtrado

Se guarda el conjunto completo 2008-2024 (réplica + extensión) ya filtrado por
geografía y causa, todavía **sin estandarizar** (nombres/categorías de variables,
grupos etarios de 7 categorías, etc. — eso es la Etapa 3).

In [ ]:
out_path = PROCESSED_DIR / "defunciones_chihuahua_suicidios_filtrado.parquet"
df_suicidios_chihuahua.to_parquet(out_path, index=False)
print(f"Guardado: {out_path} ({len(df_suicidios_chihuahua):,} filas)")

# También guardamos, aparte, las filas con discrepancia entre filtros (si las hay),
# para inspección manual sin tener que volver a correr el notebook.
if n_solo_cie10 > 0 or n_solo_tipo_defuncion > 0:
    df_discrepancias = df_chihuahua[filtro_cie10 != filtro_tipo_defuncion].copy()
    disc_path = PROCESSED_DIR / "discrepancias_filtro_causa.parquet"
    df_discrepancias.to_parquet(disc_path, index=False)
    print(f"Guardado (para revisión): {disc_path} ({len(df_discrepancias):,} filas)")

print()
print("Comparte la salida completa de este notebook (especialmente las secciones 3.3 y 4) "
      "para cerrar el reporte de la Etapa 2.")